# Elektra — Scraper por API (celulares, audífonos y smartwatches)

Sitio: https://www.elektra.mx/

Elektra es una tienda VTEX. En lugar de abrir un navegador, este notebook usa la API pública del catálogo
(`/api/catalog_system/pub/products/search`). Elektra bloquea (403, antibot Incapsula) las consultas
que no vienen de un navegador, así que se abre un Chrome oculto una sola vez y la API se consulta desde ahí.
Se filtra por categoría y **solo productos vendidos por Elektra**
(`sellerId:1`, igual que el filtro `sellername/elektra` del sitio).

Categorías:
- Telefonía › Celulares › Smartphones
- Telefonía › Accesorios para celulares › Audífonos y Manos Libres
- Telefonía › Wearables › Smartwatch

Precios por SKU (oferta del vendedor Elektra):
- `ListPrice` → precio original
- `Price` → precio de venta

In [1]:
import re
import time
from datetime import datetime

import json

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

try:
    from zoneinfo import ZoneInfo
    TZ_CDMX = ZoneInfo("America/Mexico_City")
except Exception:
    TZ_CDMX = None

# ===================== CONFIGURACIÓN =====================
SITE_URL = "https://www.elektra.mx"
API_URL = SITE_URL + "/api/catalog_system/pub/products/search"
SELLER_ID = "1"  # Elektra (excluye vendedores del marketplace)
PAGE_SIZE = 50   # máximo que permite la API por consulta

# Categoría de salida -> ruta de IDs de categoría en VTEX
CATEGORIAS = {
    "Smartphone": "1371655/1371729/1372134",  # Telefonía / Celulares / Smartphones
    "Headphone": "1371655/1371730/4843867",   # Telefonía / Accesorios para celulares / Audífonos y Manos Libres
    "Smartwatch": "1371655/1371733/1372149",  # Telefonía / Wearables / Smartwatch
}

USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36")

SCHEMA = [
    "Marca", "Nombre", "Color", "Capacidad", "SKU", "URL",
    "FechaConsulta", "Precio_original", "Precio_descuento", "Porcentaje_descuento", "Categoria",
]


In [2]:
# ===================== FUNCIONES AUXILIARES =====================
def resolver_precios(precio_lista, precio_venta, disponible):
    """Misma lógica de negocio que el resto de los scrapers."""
    if not disponible or not precio_venta:
        return "No inventory", "No discount", "No discount"
    lista = precio_lista if precio_lista and precio_lista > precio_venta else None
    if lista is None:
        return f"{precio_venta:.2f}", "No discount", "No discount"
    pct = round(100 * (lista - precio_venta) / lista, 2)
    return f"{lista:.2f}", f"{precio_venta:.2f}", f"{pct}%"


def obtener_capacidad(nombre: str, memoria_interna: str) -> str:
    """'iPhone 17 256GB' -> '256 GB'. Si el nombre no la trae, usa la especificación 'Memoria Interna'."""
    for txt in (nombre, memoria_interna):
        m = re.search(r"(\d+(?:\.\d+)?)\s*(GB|TB)", txt or "", re.IGNORECASE)
        if m:
            return f"{m.group(1)} {m.group(2).upper()}"
    return ""


def abrir_navegador():
    """Chrome oculto con una página de Elektra abierta, para pasar el antibot."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={USER_AGENT}")
    driver = webdriver.Chrome(options=options)
    driver.set_script_timeout(60)
    driver.get(SITE_URL + "/telefonia")
    time.sleep(8)
    return driver


FETCH_JS = """
const [url, cb] = [arguments[0], arguments[arguments.length - 1]];
fetch(url, {headers: {"Accept": "application/json"}})
  .then(r => r.text().then(t => cb([r.status, t])))
  .catch(e => cb([0, String(e)]));
"""


def obtener_categoria(driver, ruta: str) -> list:
    """Descarga todos los productos de una categoría vendidos por Elektra."""
    productos, desde = [], 0
    while True:
        url = (f"{API_URL}?fq=C:/{ruta}/&fq=sellerId:{SELLER_ID}"
               f"&_from={desde}&_to={desde + PAGE_SIZE - 1}")
        for intento in range(3):
            status, texto = driver.execute_async_script(FETCH_JS, url)
            if status in (200, 206):
                lote = json.loads(texto)
                break
            print(f"  Bloque {desde}, intento {intento + 1} falló: HTTP {status}")
            time.sleep(5)
        else:
            raise RuntimeError(f"No se pudo descargar el bloque {desde} de {ruta} (HTTP {status})")

        productos.extend(lote)
        if len(lote) < PAGE_SIZE:
            break
        desde += PAGE_SIZE
    return productos


def aplanar(productos: list, categoria: str, fecha: str) -> list:
    """Una fila por SKU con la oferta del vendedor Elektra."""
    filas = []
    for prod in productos:
        color = (prod.get("Color") or [""])[0]
        memoria = (prod.get("Memoria Interna") or [""])[0]
        for item in prod.get("items", []):
            oferta = next((s["commertialOffer"] for s in item.get("sellers", [])
                           if s.get("sellerId") == SELLER_ID), None)
            if oferta is None:
                continue
            nombre = (item.get("nameComplete") or prod.get("productName") or "").strip()
            p_ori, p_desc, pct = resolver_precios(
                oferta.get("ListPrice"), oferta.get("Price"), oferta.get("AvailableQuantity", 0) > 0)
            filas.append({
                "Categoria": categoria,
                "Marca": (prod.get("brand") or "").strip().upper(),
                "Nombre": nombre,
                "Color": color.strip(),
                "Capacidad": obtener_capacidad(nombre, memoria),
                "SKU": item.get("itemId", ""),
                "URL": prod.get("link", ""),
                "FechaConsulta": fecha,
                "Precio_original": p_ori,
                "Precio_descuento": p_desc,
                "Porcentaje_descuento": pct,
            })
    return filas

## Ejecutar scraping

In [3]:
ahora = datetime.now(TZ_CDMX) if TZ_CDMX is not None else datetime.now()
fecha_consulta = ahora.strftime("%Y-%m-%d %H:%M:%S")

data = []
driver = abrir_navegador()
try:
    for categoria, ruta in CATEGORIAS.items():
        productos = obtener_categoria(driver, ruta)
        filas = aplanar(productos, categoria, fecha_consulta)
        print(f"{categoria}: {len(productos)} productos -> {len(filas)} SKUs")
        data.extend(filas)
finally:
    driver.quit()

df = pd.DataFrame(data, columns=SCHEMA)
print(f"\nSKUs antes de limpiar: {len(df)}")
df = df.drop_duplicates(subset=["SKU"], keep="first")
sin_inventario = (df["Precio_original"] == "No inventory").sum()
df = df[df["Precio_original"] != "No inventory"].copy()
print(f"Excluidos sin inventario: {sin_inventario}")
# Accesorios sin capacidad (audífonos, smartwatches) -> "NA"
df.loc[(df["Categoria"] != "Smartphone") & (df["Capacidad"] == ""), "Capacidad"] = "NA"
df = df.sort_values(["Categoria", "Marca", "Nombre"]).reset_index(drop=True)
print(f"SKUs después de limpiar: {len(df)}")

Smartphone: 604 productos -> 613 SKUs


Headphone: 22 productos -> 23 SKUs


Smartwatch: 69 productos -> 71 SKUs



SKUs antes de limpiar: 707
Excluidos sin inventario: 11
SKUs después de limpiar: 696


## Exportar

In [4]:
fecha_hoy, hora_hoy = ahora.strftime("%Y-%m-%d"), ahora.strftime("%H%M%S")
xlsx_out = f"Daily Pricing Mapping Elektra ({fecha_hoy}) {hora_hoy}.xlsx"

df.to_excel(xlsx_out, index=False)
print(f"Datos exportados a {xlsx_out}")

print("\n=== ESTADÍSTICAS ===")
print(f"Total de SKUs: {len(df)}")
print("\nSKUs por categoría:")
print(df["Categoria"].value_counts())
print("\nSKUs por marca (top 15):")
print(df["Marca"].value_counts().head(15))
print(f"\nCon descuento: {(df['Porcentaje_descuento'] != 'No discount').sum()}")
display(df.head(15))

Datos exportados a Daily Pricing Mapping Elektra (2026-10-20) 093915.xlsx

=== ESTADÍSTICAS ===
Total de SKUs: 696

SKUs por categoría:
Categoria
Smartphone    605
Smartwatch     69
Headphone      22
Name: count, dtype: int64

SKUs por marca (top 15):
Marca
OPPO         118
SAMSUNG      110
APPLE        105
MOTOROLA      89
XIAOMI        82
VIVO          59
HONOR         46
REALME        21
HUAWEI        19
ZTE            9
REEBOK         8
REDMI          8
LIFEWORKS      6
KIESLECT       5
SONY           3
Name: count, dtype: int64

Con descuento: 450


,Marca,Nombre,Color,Capacidad,SKU,URL,FechaConsulta,Precio_original,Precio_descuento,Porcentaje_descuento,Categoria
0,APPLE,Audífonos Airpods 4ta Generación Apple Blancos,Blanco,NA,32011243,https://www.elektra.mx/audifonos-airpods-4ta-g...,2026-10-20 09:39:15,3499.00,No discount,No discount,Headphone
1,APPLE,Audífonos Alámbricos Apple Earpods Blancos,Blanco,NA,32011192,https://www.elektra.mx/audifonos-alambricos-ap...,2026-10-20 09:39:15,699.00,No discount,No discount,Headphone
2,LIFEWORKS,Audífonos Diadema Inalámbricos Air Buds Negros,Negro,NA,32011175,https://www.elektra.mx/audifonos-diadema-inala...,2026-10-20 09:39:15,699.00,280.00,59.94%,Headphone
3,LIFEWORKS,Audífonos Inalámbricos Air Buds Blancos,Blanco,NA,32011203,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,529.00,No discount,No discount,Headphone
4,LIFEWORKS,Audífonos Inalámbricos Air Buds Negros,Negro,NA,32011198,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,529.00,212.00,59.92%,Headphone
5,MOTOROLA,Audífonos Inalámbricos Motorola Buds 065 Negros,Negro,NA,32011184,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,729.00,292.00,59.95%,Headphone
6,MOTOROLA,Audífonos Inalámbricos Motorola Buds 065 Tofus,Blanco,NA,32011202,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,729.00,292.00,59.95%,Headphone
7,OPPO,Audífonos Inalámbricos Earbuds OPPO Enco Buds2...,Blanco,NA,32011098,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,1499.00,600.00,59.97%,Headphone
8,OPPO,Audífonos Inalámbricos OPPO Enco Buds3 Pro Bla...,Blanco,NA,32011378,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,1299.00,1169.00,10.01%,Headphone
9,OPPO,Audífonos Inalámbricos OPPO Enco Buds3 Pro Negros,Negro,NA,32011390,https://www.elektra.mx/audifonos-inalambricos-...,2026-10-20 09:39:15,1299.00,1169.00,10.01%,Headphone
